In [ ]:
## imports

import os
import chromadb
import yaml
from abc import ABC, abstractmethod
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import  MarkdownHeaderTextSplitter
from sentence_transformers import SentenceTransformer

In [ ]:
currentDirectory = os.getcwd()
currentDirectory

In [ ]:
## load data from the knowledge base 
loader = DirectoryLoader("../data/raw/",glob="**/*.md",loader_cls=TextLoader)
documents = loader.load()

print(f"number of documents :{len(documents)}")


In [ ]:
print(f"document one:{documents[0]}")
print("Content:\n", documents[0].page_content)
print("Metadata:\n", documents[0].metadata)


In [5]:
## adding extra metadata to the documents
for document in documents:
    content = document.page_content

    # Split the front matter from the Markdown content
    if content.startswith("---"):
        _, front_matter, markdown = content.split("---", 2)

        # Convert YAML front matter into a Python dictionary
        metadata = yaml.safe_load(front_matter)

        # Add your metadata to LangChain's existing metadata
        document.metadata.update(metadata)

        # Remove the metadata from the actual page content
        document.page_content = markdown

In [6]:
## creating chunks from the documents
## 1. Creating the text splitter
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "section"),
        ("##", "subsection"),
        ("###", "subsubsection")
    ]
)

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [ ]:

chunks = []

for document in documents:

    # First split according to Markdown headings
    sections = markdown_splitter.split_text(
        document.page_content
    )

    for section in sections:

        # Carry the original document metadata into the section
        section.metadata.update(document.metadata)

        # Split the section further if it is too large
        smaller_chunks = recursive_splitter.split_documents(
            [section]
        )

        for chunk in smaller_chunks:

            # Give every final chunk a unique ID
            chunk.metadata["chunk_id"] = (
                f"{document.metadata.get('document_id', 'UNKNOWN')}"
                f"-chunk-{len(chunks) + 1:03d}"
            )

            chunks.append(chunk)

print(f"Number of chunks: {len(chunks)}")

In [ ]:
print(chunks[3].page_content)
print(chunks[3].metadata)

In [9]:
class EmbeddingManager(ABC):
    @abstractmethod
    def load_embeddingmodel(self):
        pass
    @abstractmethod
    def create_embeddings(self, chunks):
        pass



In [10]:
## class implementation converting  the chunks into embeddings using sentence-transformers
class SentenceTransformerEmbeddingManager(EmbeddingManager):
    def __init__(self, model_name):
        self.model_name = model_name
        self.embedding_model = None
        

    def load_embeddingmodel(self):
      
        self.embedding_model = SentenceTransformer(self.model_name)

    def create_embeddings(self, chunks):
        if self.embedding_model is None:
            raise ValueError("Embedding model not loaded. Call load_embeddingmodel() first.")
        
        texts = [chunk.page_content for chunk in chunks]
        return self.embedding_model.encode(texts)

In [11]:
## class blueprint for the ChromaVectorStore class that will be used to store the embeddings in a 
# vector store
class ChromaVectorStore:

    def __init__(self, collection_name, persist_directory="../data/processed/vector_store"):
        self.client = chromadb.PersistentClient(
            path=persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name=collection_name
        )

    def add_chunks(self, chunks, embeddings):
        documents = [chunk.page_content for chunk in chunks]

        metadatas = [chunk.metadata for chunk in chunks]

        ids = [
            chunk.metadata["chunk_id"]
            for chunk in chunks
        ]

        self.collection.add(
            ids=ids,
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas
        )

In [ ]:
embedding_manager = SentenceTransformerEmbeddingManager(
    "all-MiniLM-L6-v2"
)

embedding_manager.load_embeddingmodel()

embeddings = embedding_manager.create_embeddings(chunks)

vector_store = ChromaVectorStore(
    collection_name="kenya_airways"
)

vector_store.add_chunks(
    chunks,
    embeddings
)